# 2D Harmonic Oscillator Field Simulation
## Physics and Mathematical Formulation

This notebook adapts the pseudo-spectral PDE solver framework to simulate a scalar field $u(x,y,t)$ subject to a 2D harmonic potential. This is the classical field equivalent of a quantum harmonic oscillator or a vibrating membrane attached to a bed of springs with spatially varying stiffness.

---

## 1. Physical Setup and Geometry

* **Domain Geometry:** A square domain $x, y \in [-L/2, L/2]$ with Dirichlet boundary conditions ($u=0$ at the edges), mimicking an infinite potential well at the boundaries.
* **The Trap:** The field experiences a restoring force proportional to its distance from the origin, creating a parabolic potential well $V(x,y) \propto (x^2 + y^2)$.

---

## 2. The Governing Equation

The system is governed by a damped Klein-Gordon-type equation (wave equation with a mass/potential term):

$$
\frac{\partial^2 u}{\partial t^2} = c^2 \nabla^2 u - \omega_0^2 (x^2 + y^2) u - \Gamma \frac{\partial u}{\partial t}
$$

Where:
* $c^2 \nabla^2 u$: Spatial wave propagation (coupling between neighboring points).
* $\omega_0^2 (x^2 + y^2) u$: The harmonic restoring force (stiffness increases with distance from the center).
* $\Gamma$: Damping coefficient (energy dissipation).

---

## 3. The Principal Symbol

In the pseudo-spectral framework, the spatial operator is defined by its principal symbol $a(x, y, \xi, \eta)$. For this system, the symbol elegantly combines Fourier space (derivatives) and physical space (potential):

$$
a(x, y, \xi, \eta) = \underbrace{c^2 (\xi^2 + \eta^2)}_{\text{Wave Propagation}} + \underbrace{\omega_0^2 (x^2 + y^2)}_{\text{Harmonic Potential}}
$$

---

## 4. Initial Conditions: Gaussian Wave Packet

To observe the oscillator's dynamics, we initialize the field with a **displaced Gaussian wave packet**. 
* In a pure harmonic oscillator, a Gaussian is the "ground state" shape. 
* By displacing it from the origin and releasing it from rest, the packet will "slosh" back and forth through the center, exhibiting classical oscillatory motion and breathing modes.

$$
u(x,y,0) = \exp\left( - \frac{(x-x_0)^2 + (y-y_0)^2}{2\sigma^2} \right), \quad \frac{\partial u}{\partial t}(x,y,0) = 0
$$

# Implementation
## 0. Imports

In [ ]:
from solver import PDESolver, psiOp
import sympy as sp
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import HTML

## 1. Physical and simulation parameters

In [ ]:
# ── Wave speed and Oscillator frequency ──
C_SQUARED       = 0.5   # c² (m²/s²); spatial coupling / wave propagation speed
OMEGA_0_SQUARED = 0.3   # ω₀² (s⁻²); strength of the harmonic potential (stiffness)

# ── Dissipation ──
GAMMA = 0.0            # Damping coefficient (s⁻¹); 0 = perpetual oscillation

# ── Grid and Time ──
Lx, Ly   = 10.0, 10.0
# Nx, Ny = 32, 32 
Nx, Ny = 64, 64    
# Nx, Ny = 128, 128    
# Nx, Ny = 128, 256    

Lt, Nt   = 20.0, 400
# Lt, Nt   = 30.0, 600
# Lt, Nt   = 40.0, 800
# Lt, Nt   = 50.0, 1000
# Lt, Nt   = 60.0, 1200
n_frames = 200

## 2. Grid setup

In [ ]:
xs_1d = np.linspace(-Lx/2, Lx/2, Nx)
ys_1d = np.linspace(-Ly/2, Ly/2, Ny)

# CRITICAL FIX: psipy internally uses indexing='ij' (axis 0 = x, axis 1 = y).
# We MUST match this convention so the solver receives the correct array orientation.
xx, yy = np.meshgrid(xs_1d, ys_1d, indexing='ij')   # shape (Nx, Ny)

## 3. SymPy symbols and principal symbol

In [ ]:
x, y, t = sp.symbols('x y t', real=True)
xi, eta = sp.symbols('xi eta', real=True)
u_func  = sp.Function('u')
u       = u_func(t, x, y)

# Full principal symbol:
#   a(x, y, ξ, η) = c²·(ξ² + η²)           ← wave propagation (Fourier space)
#                 + ω₀²·(x² + y²)           ← harmonic potential (Physical space)
symbol_wave     = C_SQUARED * (xi**2 + eta**2)
symbol_harmonic = OMEGA_0_SQUARED * (x**2 + y**2)

symbol_num = symbol_wave + symbol_harmonic

print("Principal symbol:")
print("  a(x, y, ξ, η) =", symbol_num)

## 4. Wave equation

In [ ]:
#
#   ∂²u/∂t² = -psiOp(a(ξ), u) - GAMMA·∂u/∂t
#
gamma    = sp.Symbol('gamma', positive=True)
equation = sp.Eq(
    sp.diff(u, t, 2),
    -psiOp(symbol_num, u) - gamma * sp.diff(u, t)
)
equation_num = equation.subs({gamma: GAMMA})

print("Equation:")
print(f"  ∂²u/∂t² = -psiOp({symbol_num}, u) - {GAMMA}·∂u/∂t")

## 5. Initial conditions

In [ ]:
def initial_condition_ho(xx, yy):
    """
    Displaced Gaussian wave packet.
    Released from an offset position to trigger oscillation.
    """
    x0, y0 = 0.0, 0.0  # Initial displacement from the center
    sigma = 0.8        # Width of the packet
    return np.exp(-((xx - x0)**2 + (yy - y0)**2) / (2 * sigma**2))

def initial_velocity_ho(xx, yy):
    """
    Released from rest.
    """
    return np.zeros_like(xx)

## 6. Solver setup

In [ ]:
solver = PDESolver(equation_num)

solver.setup(
    Lx=Lx, Ly=Ly,
    Nx=Nx, Ny=Ny,
    Lt=Lt, Nt=Nt,
    boundary_condition='dirichlet', # Dirichlet (u=0) at boundaries for harmonic trap
    initial_condition=initial_condition_ho,
    initial_velocity=initial_velocity_ho,
    n_frames=n_frames,
    plot=True,
)

## 7. Solve

In [ ]:
frames = solver.solve()

## 8. Visualization

In [ ]:
# Raise the animation size limit to allow large frame counts at high resolution
plt.rcParams['animation.embed_limit'] = 2**128

ani = solver.animate(
    component='real',
    overlay='contour', # Contours beautifully highlight the harmonic modes
    mode='surface',    # 'imshow' or 'surface'
    physical=True      # True or False
)

HTML(ani.to_jshtml())

In [ ]:
ani.save('harmonic_oscillator_2d.mp4', writer='ffmpeg', fps=20, dpi=100)
print("✅ Saved to harmonic_oscillator_2d.mp4")